# 03 — Snapshot-padding Transformer training with L1 and L2

All 49 predictors come from already encoded V63 `MODEL_DATA` on the exact `(PATIENT_ID, END_DT)` key. The same snapshot vector is repeated only at enrollment-valid positions; all other positions are zero and masked. No historical feature values are reconstructed and no monthly feature values are aggregated into quarterly values.

MONTHLY is `[N, 12, 49]`; QUARTERLY is `[N, 4, 49]`. The newest position is index 0. A quarterly position is valid exactly when any of its three corresponding monthly positions is valid. Any inclusive enrollment-interval overlap is sufficient, including one day; this does not assert full-period or separate MX/RX coverage.

Run notebook 01 first, review its source/coverage audits, and copy the printed `EXPECTED_PREPARED_MANIFEST_SHA256` into 02–04. Use the fresh identifiers `D20260927S` / `R20260927S` and output namespace `TAK861_TX_READY_V63_DL_POC_SNAPSHOT_PAD_V1`. There are no dependencies on old experimental tensors, models or run reports. The frozen snapshot and patient-split tables remain the requested source population.

Imputation/scaling is fitted once per eligible TRAIN snapshot, then the identical state is used for both representations. Thus more covered periods do not increase a snapshot's preprocessing weight. This is still snapshot weighting, not one-record-per-patient training. All-padded snapshots remain in model training and reported populations, but contribute no unavailable feature values to the preprocessing fit.

The comparison measures sensitivity to sequence length, positional parameters and enrollment masks on repeated snapshot values. It does not establish temporal feature learning, a quarterly aggregation advantage, or the causal benefit of padding. Upstream V63 encoding used RESP and its original fit population is not verified against the frozen held-out split; results remain retrospective. Keep credentials and row-level artifacts private and clear outputs before sharing.


### Configuration and experiment identity

**Why:** Select the fresh snapshot-padding experiment (`D20260927S` / `R20260927S`, namespace `SNAPSHOT_PAD_V1`) and its exact prepared-input fingerprint. Set the manifest hash printed by notebook 01 before running dependent cells; keep identifiers and settings consistent across notebooks.

**Expected output:** This cell defines configuration and normally displays nothing. Missing secure connection settings or invalid identifiers stop execution.

**Interpretation:** MONTHLY and QUARTERLY contain the same 49 encoded snapshot features repeated over 12 or 4 eligible positions; they are different enrollment-mask views, not reconstructed histories or quarterly feature aggregates. L1/L2 and network settings control training, not a manually assigned feature priority.

In [ ]:
# Use the same fresh identifiers and reviewed manifest hash in 01–04.
import os
import re
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
if 'sf_options' not in globals() or not isinstance(sf_options, dict) or 'spark' not in globals():
    raise RuntimeError('Supply sf_options securely on the approved Spark runtime; never paste credentials into this notebook.')
DATABASE = 'DSVC_TAKEDA_TA_PRIVATE'
sf_options_dl_poc = dict(sf_options)
sf_options_dl_poc.update(sfDatabase=DATABASE, sfSchema='DS_ML')
SOURCE_PREFIX = 'TAK861_TX_READY_V63'
PREFIX = SOURCE_PREFIX + '_DL_POC'
DATASET_ID = 'D20260927S'
RUN_ID = 'R20260927S'
EXPECTED_PREPARED_MANIFEST_SHA256 = None  # Set only after reviewing the clean notebook-01 export.
if any(not re.fullmatch(r'[A-Z][A-Z0-9_]{0,15}', v) for v in (DATASET_ID, RUN_ID)):
    raise ValueError('Use matching short uppercase dataset/run identifiers in all notebooks.')
EXPERIMENT_PREFIX = PREFIX + '_SNAPSHOT_PAD_V1'
PREPARED_TABLE = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_INPUTS'
SPLIT_TABLE = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_SPLIT'
RUN_PREFIX = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_' + RUN_ID
PREPARED_NAMES = {'population.json', 'manifest.json', 'MONTHLY.npz', 'QUARTERLY.npz'}
SPLIT_NAMES = {'split.json', 'preprocessing.json', 'audit.json'}
MODEL_NAMES = {'checkpoint.pt', 'summary.json', 'history.json'}
MODEL_SETTINGS = dict(d_model=128, n_heads=4, encoder_layers=2, feedforward_dim=256, dropout=.2)
TRAINING_SETTINGS = dict(seed=42, epochs=20, patience=5, min_delta=1e-4, batch_size=64,
                        learning_rate=.001, weight_decay=0., grad_clip=1., device='auto',
                        l1_lambda=1e-6, l2_lambda=1e-4,
                        regularization_scope='linear_and_attention_weights')
CONSUMER_IMPLEMENTATION_SHA256 = 'a06e90d8e6ee4b87b6cba1503ee7ed3c3b580db9c4d8b7c2c87c60ea7d8cf4cb'


### Runtime libraries and artifact layout

**Why:** Load the libraries needed by subsequent cells and declare the columns used to store verified artifacts in the private warehouse. Imports define capabilities; they do not query patient data or train a model.

**Expected output:** Normally none. An import failure means the approved runtime needs that dependency before continuing.

**Interpretation:** A successful import only confirms library availability. It does not validate sources, tensors, feature rankings, or model performance. Any mask convention declared here uses `valid=True` for an enrollment-eligible position; eligibility does not establish that the repeated feature value was observed historically in that period.

In [ ]:
import io
import json
import hashlib
import numpy as np
import pandas as pd
import base64
ARTIFACT_COLUMNS = ["ARTIFACT_NAME", "CHUNK_INDEX", "CHUNK_COUNT", "BYTE_LENGTH", "SHA256", "PAYLOAD_BASE64"]
import math
from dataclasses import dataclass
import torch
from torch import nn
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)
import platform
import random
import sklearn
from dataclasses import asdict
from datetime import datetime, timezone
from torch.utils.data import Dataset, DataLoader
TIME_ORIENTATION = "0=most_recent; increasing_index=older"
MASK_CONVENTION = "valid=True means observed; src_key_padding_mask=~valid; masked mean pooling"


### Verified artifact read/write helpers

**Why:** Define source reads and chunked artifact storage with length and SHA-256 checks. These checks connect downstream calculations to the exact inputs produced by notebook 01 and prevent silently replacing a different result.

**Expected output:** None while these functions are defined. Later saves report successful write/read-back verification or stop on a mismatch.

**Interpretation:** Matching hashes establish artifact integrity, not clinical correctness or absence of modeling bias. An existing destination with different content is an experiment identity conflict; use a deliberately new identifier after reviewing the change. This rebuild never needs a checkpoint from a previous experiment.

In [ ]:
def require(condition, message):
    if not condition:
        raise ValueError(message)


def array_hash(values):
    values = np.asarray(values)
    h = hashlib.sha256(canonical_json(list(values.shape)).encode())
    h.update(np.isnan(values).astype('u1').tobytes())
    h.update(np.nan_to_num(values, nan=0).astype('<f8').tobytes())
    return h.hexdigest()


def canonical_json(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)


def digest_json(value):
    return hashlib.sha256(canonical_json(value).encode()).hexdigest()


def read_table(table):
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load())


def table_exists(table):
    if not re.fullmatch(r"[A-Z][A-Z0-9_]*", table):
        raise ValueError("Use uppercase letters, numbers and underscores in table names.")
    query = ("SELECT TABLE_NAME FROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.TABLES "
             f"WHERE TABLE_SCHEMA = 'DS_ML' AND TABLE_NAME = '{table}'")
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("query", query).load().limit(1).count() > 0)


def pack_artifacts(artifacts, chunk_size=50000):
    rows = []
    for name, blob in artifacts.items():
        encoded = base64.b64encode(blob).decode("ascii")
        pieces = [encoded[i:i + chunk_size] for i in range(0, len(encoded), chunk_size)] or [""]
        digest = hashlib.sha256(blob).hexdigest()
        rows.extend((name, i, len(pieces), len(blob), digest, piece)
                    for i, piece in enumerate(pieces))
    return rows


def unpack_artifacts(rows, expected_names):
    groups = {}
    for row in rows:
        name, i, count, size, digest, payload = tuple(row)
        if any(value != int(value) for value in (i, count, size)):
            raise ValueError("Nonintegral artifact chunk metadata.")
        groups.setdefault(name, []).append((int(i), int(count), int(size), digest, payload))
    if set(groups) != set(expected_names):
        raise ValueError("Missing or unexpected saved artifacts.")
    result = {}
    for name, pieces in groups.items():
        pieces.sort(key=lambda p: p[0])
        count, size, digest = pieces[0][1:4]
        if (count < 1 or size < 0 or len(pieces) != count
                or [p[0] for p in pieces] != list(range(count))
                or any(p[1:4] != (count, size, digest) for p in pieces)):
            raise ValueError("Missing, duplicate or inconsistent artifact chunks.")
        blob = base64.b64decode("".join(p[4] for p in pieces), validate=True)
        if len(blob) != size or hashlib.sha256(blob).hexdigest() != digest:
            raise ValueError("Artifact length/hash mismatch.")
        result[name] = blob
    return result


def read_artifacts(table, expected_names):
    rows = (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load().select(*ARTIFACT_COLUMNS).collect())
    return unpack_artifacts(rows, expected_names)


def save_artifacts(table, artifacts):
    # A matching existing result can be verified after an interrupted read-back.
    if table_exists(table):
        if read_artifacts(table, artifacts) != artifacts:
            raise FileExistsError("Destination contains different artifacts; choose a new RUN_ID.")
        print(f"Existing artifacts verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")
        return
    schema = ("ARTIFACT_NAME STRING, CHUNK_INDEX INT, CHUNK_COUNT INT, "
              "BYTE_LENGTH LONG, SHA256 STRING, PAYLOAD_BASE64 STRING")
    frame = spark.createDataFrame(pack_artifacts(artifacts), schema=schema)
    (frame.write.format("snowflake").options(**sf_options_dl_poc)
     .option("dbtable", table).option("truncate_columns", "off")
     .mode("errorifexists").save())
    if read_artifacts(table, artifacts) != artifacts:
        raise ValueError("Saved artifact read-back differs from the completed run.")
    print(f"Saved and verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")


### Authoritative features and frozen V63 encoding

**Why:** Require all 49 feature names, in the configured order, to agree between `MODEL_TYPE.FEATURES`, `FINAL_MODEL.SEQ`, and the preparation manifest. Validate each frozen encoding parameter and the source audit.

**Expected output:** None on definition. A later mismatch displays the conflicting contract and raises an error.

**Interpretation:** All features in this experiment must be `V63_SNAPSHOT`, joined to the exact patient/date and broadcast only to eligible positions. Existing `MODEL_DATA` values are already encoded; do not apply the business transformation again. Frozen upstream encoding was not verified as fitted on TRAIN alone, so this remains a retrospective limitation. Feature order and encoding caps do not rank feature importance.

In [ ]:
def parse_features(value):
    if isinstance(value, str):
        value = json.loads(value)
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if not isinstance(value, list) or not value or any(not isinstance(x, str) or not x.strip() or x != x.strip() for x in value):
        raise ValueError("FEATURES must be a nonempty array of exact column names.")
    if len(set(x.upper() for x in value)) != len(value):
        raise ValueError("Duplicate feature names.")
    prohibited = {"PATIENT_ID", "START_DT", "END_DT", "RESP", "SPLIT", "RND", "SCORE", "DECILE", "CENTILE", "MILLILE"}
    if prohibited.intersection(x.upper() for x in value):
        raise ValueError("The selected list contains an identifier, target, split, random helper or prediction output.")
    return value


def _v63_require(condition, message):
    if not condition:
        raise ValueError(message)


def _v63_features(features):
    _v63_require(isinstance(features, (list, tuple, np.ndarray)),
                 'Authoritative features must be an ordered sequence.')
    result = list(features)
    _v63_require(0 < len(result) <= 49, 'Retained predictors must be a nonempty subset of the 49 candidates.')
    _v63_require(all(isinstance(name, str) and name and name == name.strip()
                     for name in result), 'Feature names must be exact, nonempty strings.')
    _v63_require(len({name.upper() for name in result}) == len(result),
                 'Duplicate authoritative features are not permitted.')
    prohibited = {'PATIENT_ID', 'END_DT', 'START_DT', 'RESP', 'SPLIT',
                  'SCORE', 'DECILE', 'CENTILE', 'MILLILE', 'RND'}
    _v63_require(not prohibited.intersection(name.upper() for name in result),
                 'Identifiers, labels and prediction outputs cannot be predictors.')
    return result


def _v63_cap(value, feature):
    _v63_require(not isinstance(value, (bool, np.bool_)),
                 feature + ': VALUE_P must be numeric, not boolean.')
    _v63_require(not isinstance(value, (complex, np.complexfloating)),
                 feature + ': VALUE_P must be real.')
    try:
        cap = float(value)
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError(feature + ': invalid VALUE_P.') from error
    _v63_require(math.isfinite(cap) and cap >= 0,
                 feature + ': VALUE_P must be finite and nonnegative.')
    return cap


def _v63_type(value, feature):
    _v63_require(isinstance(value, str), feature + ': VAR_TYP must be a declared type.')
    normalized = value.strip().upper()
    _v63_require(normalized in {'BINARY', 'NUMERIC'},
                 feature + ': VAR_TYP must be Binary or Numeric; Dropped/unknown types cannot be silently removed.')
    return {'BINARY': 'Binary', 'NUMERIC': 'Numeric'}[normalized]


def _v63_parameter_hash(records):
    encoded = json.dumps(records, sort_keys=True, ensure_ascii=False,
                         separators=(',', ':'), allow_nan=False).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()


def fixed_v63_parameters(features, summary):
    """Return (selected ordered parameter records, SHA256) from frozen summary rows.

    FEATURES_SUMMARY may contain additional rows; they do not alter the explicit
    selected feature list. Every summary feature name must be unique, and every
    authoritative feature must have exactly one valid parameter row. No row is
    chosen by importance, VALUE_P magnitude, rank, type, or observed input values.
    """
    features = _v63_features(features)
    _v63_require(isinstance(summary, pd.DataFrame), 'FEATURES_SUMMARY must be a DataFrame.')
    required = ['FEATURES', 'VALUE_P', 'VAR_TYP']
    _v63_require(set(required).issubset(summary.columns),
                 'FEATURES_SUMMARY must provide FEATURES, VALUE_P and VAR_TYP.')
    _v63_require(not summary.columns.duplicated().any(), 'Duplicate summary columns are invalid.')
    names = summary['FEATURES'].tolist()
    _v63_require(all(isinstance(name, str) and name and name == name.strip() for name in names),
                 'Summary feature names must be exact, nonempty strings.')
    _v63_require(len({name.upper() for name in names}) == len(names),
                 'Duplicate FEATURES_SUMMARY feature rows are not permitted.')
    indexed = summary.set_index('FEATURES', verify_integrity=True)
    missing = [name for name in features if name not in indexed.index]
    _v63_require(not missing, 'Missing fixed parameter rows: ' + ', '.join(missing))
    records = []
    for order, name in enumerate(features):
        row = indexed.loc[name]
        cap = _v63_cap(row['VALUE_P'], name)
        kind = _v63_type(row['VAR_TYP'], name)
        transform = ('AGE_CEIL_DECADE' if name.upper() == 'AGE'
                     else 'BINARY_POSITIVE' if kind == 'Binary'
                     else 'NUMERIC_FIXED_CAP')
        records.append({'FEATURE_ORDER': order, 'FEATURES': name, 'VALUE_P': cap,
                        'VAR_TYP': kind, 'TRANSFORM': transform})
    return records, _v63_parameter_hash(records)


def configured_features():
    config = read_table(SOURCE_PREFIX + '_MODEL_TYPE').select('MODEL_TYPE', 'FEATURES').collect()
    require(len(config) == 1, 'Expected one frozen V63 MODEL_TYPE configuration.')
    features = parse_features(config[0]['FEATURES'])
    require(len(features) == 49, 'Exactly 49 V63 model predictors are required.')
    final = read_table(SOURCE_PREFIX + '_FINAL_MODEL').select('FEATURES', 'SEQ').toPandas()
    require(not final[['FEATURES', 'SEQ']].isna().any().any(), 'FINAL_MODEL has missing names/order.')
    require(not final.FEATURES.duplicated().any() and not final.SEQ.duplicated().any(), 'FINAL_MODEL has duplicate feature/order rows.')
    final['SEQ'] = pd.to_numeric(final.SEQ, errors='raise')
    require(np.isfinite(final.SEQ).all() and final.SEQ.eq(np.floor(final.SEQ)).all(), 'FINAL_MODEL SEQ must be finite integer ranks.')
    final_features = final.sort_values('SEQ').FEATURES.tolist()
    comparison = {'configured_model_type': str(config[0]['MODEL_TYPE']), 'configured_feature_count': len(features),
        'final_model_feature_count': len(final_features), 'configured_not_in_final_model': sorted(set(features) - set(final_features)),
        'final_model_not_in_configuration': sorted(set(final_features) - set(features)),
        'final_model_order_matches': features == final_features,
        'ordering_rule': 'MODEL_TYPE.FEATURES retained; FINAL_MODEL ORDER BY SEQ must agree before constructing tensors',
        'feature_role': 'Frozen candidate contract; retained subset declared by preparation manifest'}
    if features != final_features:
        display(pd.DataFrame([comparison]))
        display(pd.DataFrame({'MODEL_TYPE_ORDER': pd.Series(features), 'FINAL_MODEL_SEQ_ORDER': pd.Series(final_features)}))
        raise ValueError('V63 feature lists/order disagree. Resolve source discrepancy; do not silently reorder or select features.')
    return features, comparison


def load_business_parameters(features):
    summary = read_table(SOURCE_PREFIX + '_FEATURES_SUMMARY').toPandas()
    parameters, parameter_hash = fixed_v63_parameters(features, summary)
    contract = {'version': 1, 'parameters': parameters, 'parameter_sha256': parameter_hash,
        'source': DATABASE + '.DS_ML.' + SOURCE_PREFIX + '_FEATURES_SUMMARY',
        'input_space': 'already encoded V63 MODEL_DATA snapshot values', 'encoding_applied_here': False, 'output_space': 'V63 MODEL_DATA business encoding',
        'snapshot_MODEL_DATA_already_encoded': True, 'caps_or_types_refitted_here': False,
        'upstream_fitting_uses_RESP': True,
        'upstream_fit_population': 'Not verified against frozen TRAIN/VALIDATION/TEST; fixed parameter reuse is retrospective',
        'formula': 'AGE ceil(raw/10); Binary raw>0; Numeric raw>VALUE_P => 1 else raw/(VALUE_P+1)'}
    return parameters, summary, contract


def selected_features(manifest):
    candidates, _ = configured_features()
    require(manifest.get('source_features') == candidates, 'Frozen candidate names/order changed.')
    features = parse_features(manifest.get('features'))
    require(features == candidates, 'Snapshot-only input must retain all frozen candidates in their original order.')
    exclusions = manifest.get('excluded_features')
    require(isinstance(exclusions, list) and all(isinstance(r, dict) for r in exclusions),
            'An explicit exclusion audit is required, including an empty list when none are excluded.')
    require([r.get('FEATURE_NAME') for r in exclusions] == [f for f in candidates if f not in features],
            'Every excluded candidate must be accounted for once, in original order.')
    require(all(isinstance(r.get('REASON'), str) and r['REASON'].strip() for r in exclusions),
            'Each excluded feature needs a specific reason.')
    audit = manifest.get('audit')
    require(isinstance(audit, list) and all(isinstance(r, dict) for r in audit), 'Missing retained-feature audit.')
    require([r.get('FEATURE_NAME') for r in audit] == features, 'Retained feature audit is misordered.')
    for row in audit:
        require(row.get('STATUS') == 'IMPLEMENTED_VALIDATED',
                'Each feature extraction path must be implemented and validated.')
        require(row.get('VALUE_SOURCE') == 'V63_SNAPSHOT', 'Every feature must declare V63_SNAPSHOT in this snapshot-only experiment.')
        for key in ('SOURCE', 'CALCULATION', 'LOOKBACK', 'COVERAGE_RULE', 'TEMPORAL_JUSTIFICATION', 'VALIDATION'):
            require(isinstance(row.get(key), str) and bool(row[key].strip()),
                    row['FEATURE_NAME'] + ': missing audit field ' + key)
        if row['VALUE_SOURCE'] == 'V63_SNAPSHOT':
            require(row.get('SOURCE') == DATABASE + '.DS_ML.' + SOURCE_PREFIX + '_MODEL_DATA',
                    'Snapshot fallback must use the matching V63 MODEL_DATA source.')
            require(row.get('SNAPSHOT_JOIN') == ['PATIENT_ID', 'END_DT']
                    and row.get('ALREADY_ENCODED') is True
                    and row.get('BROADCAST') == 'valid_timesteps_only',
                    'Snapshot fallback must match the exact snapshot, stay encoded and respect padding.')
            require(isinstance(row.get('FALLBACK_REASON'), str) and bool(row['FALLBACK_REASON'].strip()),
                    'Snapshot fallback requires a reason.')
    return features


### Prepared inputs, cohort, and split validation helpers

**Why:** Define checks for the reviewed manifest, tensor shapes, zero padding, source hashes, exact snapshot keys/labels, and frozen patient assignments. The snapshot values must be constant across every eligible position and agree between representations.

**Expected output:** No result table until these helpers are called. Invalid inputs stop execution instead of being silently repaired.

**Interpretation:** The expected cohort is 23,151 snapshots, 12,447 patients, and 1,345 positive snapshots. MONTHLY has 12 positions; QUARTERLY has 4. These names describe enrollment buckets only: no historical feature reconstruction or averaging/summing of monthly feature values occurs. Patient separation prevents the same patient crossing partitions; it cannot remove upstream encoding limitations.

In [ ]:
def exact_dates(values, name):
    require(not values.isna().any(), name + ': missing dates are not allowed.')
    require(not values.map(lambda v: isinstance(v, (bool, np.bool_, int, float, np.number))).any(),
            name + ': numeric/Boolean values cannot be interpreted as dates.')
    dates = pd.to_datetime(values, errors='raise')
    require(dates.dt.tz is None and dates.eq(dates.dt.normalize()).all(), name + ': expected exact dates without timezone/time of day.')
    return dates.dt.strftime('%Y-%m-%d')


def snapshot_records(metadata):
    return [[r.PATIENT_ID, r.END_DT, int(r.RESP)] for r in metadata.itertuples()]


def require_prepared_selection(manifest=None):
    """Consume only the explicitly reviewed output of the clean preparation notebook."""
    require(isinstance(EXPECTED_PREPARED_MANIFEST_SHA256, str)
            and re.fullmatch(r'[0-9a-f]{64}', EXPECTED_PREPARED_MANIFEST_SHA256) is not None,
            'Set EXPECTED_PREPARED_MANIFEST_SHA256 to the reviewed notebook-01 manifest hash in 02/03/04.')
    if manifest is not None:
        require(digest_json(manifest) == EXPECTED_PREPARED_MANIFEST_SHA256,
                'Prepared manifest differs from the reviewed output. Do not automatically change the pin.')


def load_prepared():
    require_prepared_selection()
    blobs = read_artifacts(PREPARED_TABLE, PREPARED_NAMES)
    manifest = json.loads(blobs['manifest.json'])
    require_prepared_selection(manifest)
    require(manifest.get('schema') == 5 and manifest.get('dataset_id') == DATASET_ID,
            'Prepared dataset ID/schema changed.')
    require(manifest.get('padding_contract') == 'independent_timestep_availability_v1'
            and manifest.get('feature_policy') == 'v63_snapshot_only_valid_timestep_broadcast_v1', 'Unexpected preparation contract.')
    require(manifest.get('source_vintage') == '20260825', 'Unexpected source vintage.')
    require(isinstance(manifest.get('implementation_sha256'), str)
            and re.fullmatch(r'[0-9a-f]{64}', manifest['implementation_sha256']) is not None,
            'Missing preparation implementation fingerprint.')
    require(manifest.get('orientation') == '0=newest; snapshot-relative month buckets; newest ends at END_DT',
            'Prepared timestep orientation changed.')
    require(manifest.get('bucket_definition') == {
        'MONTHLY': {'steps': 12, 'months_per_step': 1},
        'QUARTERLY': {'steps': 4, 'months_per_step': 3},
        'anchor': 'snapshot calendar month',
        'start': 'first day of anchor month minus width*step minus (width-1) months',
        'end': 'min(END_DT, last day of anchor month minus width*step months)'},
        'Bucket definition changed; quarterly buckets must align with monthly triples.')
    features = selected_features(manifest)
    require(manifest.get('preprocessing_policy') == 'one_vector_per_eligible_train_snapshot_shared_v1', 'Unexpected preprocessing policy.')
    require(isinstance(manifest.get('frozen_split_assignments_sha256'), str)
            and re.fullmatch(r'[0-9a-f]{64}', manifest['frozen_split_assignments_sha256']) is not None,
            'Missing frozen split assignment fingerprint.')
    require(manifest.get('input_description') == 'V63 snapshot features repeated only at enrollment-valid timesteps; no historical feature reconstruction',
            'Input description must identify the snapshot-only repeated values.')
    _, _, encoding = load_business_parameters(features)
    require(encoding == manifest.get('business_encoding'), 'Frozen V63 encoding changed after preparation.')
    population = json.loads(blobs['population.json'])
    metadata = pd.DataFrame(population, columns=['PATIENT_ID', 'END_DT', 'RESP'])
    metadata = normalize_metadata(metadata).sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    require(population == snapshot_records(metadata), 'Prepared snapshot order changed.')
    population_check(metadata)
    source = normalize_metadata(read_table(PREFIX + '_SNAPSHOTS').select('PATIENT_ID', 'END_DT', 'RESP').toPandas())
    source = source.sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    require(metadata.equals(source), 'Frozen population/RESP changed after preparation.')
    require(digest_json(population) == manifest.get('population_sha256'), 'Population hash mismatch.')
    require(set(manifest.get('representations', {})) == {'MONTHLY', 'QUARTERLY'}, 'Both representations required.')
    bundles = {}
    for name, count in (('MONTHLY', 12), ('QUARTERLY', 4)):
        with np.load(io.BytesIO(blobs[name + '.npz']), allow_pickle=False) as arrays:
            require(set(arrays.files) == {'X', 'valid'}, 'Unexpected prepared array names.')
            X, valid = arrays['X'].copy(), arrays['valid'].copy()
        require(X.dtype == np.float32 and X.shape == (len(metadata), count, len(features))
                and valid.shape == X.shape[:2] and valid.dtype == bool, 'Invalid tensor or mask dimensions/type.')
        require(not np.isinf(X).any() and np.isfinite(X[~valid]).all() and np.all(X[~valid] == 0),
                'Invalid padding/infinite input; individual feature NaNs belong only to valid timesteps.')
        info = manifest['representations'][name]
        coverage = info.get('coverage_audit', {})
        require(coverage.get('unknown_timesteps') == 0, 'Unknown coverage must be resolved before preparation.')
        require(coverage.get('available_timesteps') == int(valid.sum())
                and coverage.get('unavailable_timesteps') == int((~valid).sum())
                and coverage.get('all_padded_snapshots') == int((~valid.any(axis=1)).sum()),
                'Coverage audit disagrees with the saved mask.')
        require(isinstance(coverage.get('policy'), str) and bool(coverage['policy'].strip())
                and isinstance(coverage.get('evidence'), str) and bool(coverage['evidence'].strip()),
                'Independent source-coverage evidence and policy are required.')
        require(digest_json(coverage) == info.get('availability_sha256'), 'Coverage audit fingerprint changed.')
        require(isinstance(info.get('provenance'), list) and len(info['provenance']) > 0,
                'Missing source/builder provenance.')
        require(array_hash(X) == info.get('X_sha256') and array_hash(valid) == info.get('valid_sha256'),
                'Tensor/mask fingerprint changed.')
        require(info.get('shape') == list(X.shape) and info.get('value_space') == 'V63 encoded',
                'Unexpected tensor shape/value space; do not re-encode upstream values.')
        for feature_index, row in enumerate(manifest['audit']):
            if row['VALUE_SOURCE'] == 'V63_SNAPSHOT':
                first_valid = valid.argmax(axis=1)
                reference = X[np.arange(len(X)), first_valid, feature_index][:, None]
                repeated = np.broadcast_to(reference, valid.shape)
                require(np.array_equal(X[:, :, feature_index][valid], repeated[valid], equal_nan=True),
                        row['FEATURE_NAME'] + ': snapshot fallback varies within a snapshot.')
        bundles[name] = {'raw_X': X, 'valid': valid, 'representation': name}
    monthly, quarterly = bundles['MONTHLY'], bundles['QUARTERLY']
    require(np.array_equal(quarterly['valid'], monthly['valid'].reshape(len(metadata), 4, 3).any(axis=2)),
            'Quarterly validity must be OR over the three corresponding monthly positions.')
    both = monthly['valid'].any(axis=1) & quarterly['valid'].any(axis=1)
    rows = np.flatnonzero(both)
    for feature_index, row in enumerate(manifest['audit']):
        if row['VALUE_SOURCE'] == 'V63_SNAPSHOT':
            a = monthly['raw_X'][rows, monthly['valid'][rows].argmax(axis=1), feature_index]
            b = quarterly['raw_X'][rows, quarterly['valid'][rows].argmax(axis=1), feature_index]
            require(np.array_equal(a, b, equal_nan=True), 'Snapshot fallback differs between representations.')
    return metadata, features, bundles, manifest


def split_statistics(metadata):
    out = []
    sets = {name: set(metadata.loc[metadata.SPLIT.eq(name), 'PATIENT_ID']) for name in ('train', 'validation', 'test')}
    expected = {'train': (16256, 8712, 941), 'validation': (3481, 1867, 202), 'test': (3414, 1868, 202)}
    for name, ids in sets.items():
        part = metadata.loc[metadata.SPLIT.eq(name)]
        require((len(part), len(ids), int(part.RESP.sum())) == expected[name], 'Original split counts changed: ' + name)
        out.append({'SPLIT': name, 'PATIENTS': len(ids), 'SNAPSHOTS': len(part), 'RESP_0': int(part.RESP.eq(0).sum()),
                    'RESP_1': int(part.RESP.sum()), 'POSITIVE_RATE': float(part.RESP.mean())})
    for a, b in (('train', 'validation'), ('train', 'test'), ('validation', 'test')):
        require(not sets[a].intersection(sets[b]), 'Patient overlap: ' + a + '/' + b)
    return pd.DataFrame(out)


def normalize_metadata(frame):
    require(not frame.columns.duplicated().any(), 'Duplicate metadata columns.')
    out = frame[['PATIENT_ID', 'END_DT', 'RESP']].copy().reset_index(drop=True)
    require(not out.empty and not out.isna().any().any(), 'Missing snapshot keys or labels.')
    require(out.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x) and x == x.strip()).all(),
            'Patient IDs must remain exact nonempty strings without surrounding whitespace.')
    out['END_DT'] = exact_dates(out.END_DT, 'END_DT')
    require(not out.RESP.map(lambda x: isinstance(x, (str, bool, np.bool_))).any()
            and out.RESP.isin([0, 1]).all(), 'RESP must be numeric 0 or 1, without coercion.')
    out['RESP'] = out.RESP.astype('int64')
    require(not out.duplicated(['PATIENT_ID', 'END_DT']).any(), 'Duplicate patient/date keys; no automatic deduplication.')
    return out


def bind_split(metadata, frozen):
    original = normalize_metadata(metadata).sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    normalized = normalize_metadata(frozen)
    normalized['SPLIT'] = frozen.SPLIT.to_numpy()
    normalized = normalized.sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    require(original.equals(normalized[['PATIENT_ID', 'END_DT', 'RESP']]),
            'Frozen split differs from the prepared snapshots/labels; exact set equality is required.')
    require(not normalized.SPLIT.isna().any() and set(normalized.SPLIT) == {'train', 'validation', 'test'},
            'Unexpected or missing frozen split labels.')
    require(not normalized.groupby('PATIENT_ID').SPLIT.nunique().gt(1).any(), 'Patient leakage between splits.')
    records = [[r.PATIENT_ID, r.END_DT, int(r.RESP), r.SPLIT] for r in normalized.itertuples()]
    hashes = {'snapshot_manifest_sha256': digest_json(records),
              'assignment_contract': 'exact_frozen_patient_split_v1'}
    split_statistics(normalized)
    return normalized, hashes


def population_check(metadata):
    observed = (len(metadata), metadata.PATIENT_ID.nunique(), int(metadata.RESP.sum()))
    if observed != (23151, 12447, 1345):
        raise ValueError(f"Frozen V63 cohort changed: snapshots/patients/positives = {observed}")


### One-value-per-snapshot TRAIN preprocessing

**Why:** Define median imputation and standardization using one feature vector per eligible TRAIN snapshot, taken from its first valid position. Share the resulting frozen statistics across MONTHLY and QUARTERLY, apply them to other partitions, and leave padded positions exactly zero.

**Expected output:** None while functions are defined. Later calls fail if a feature has no finite eligible TRAIN observation or if the shared saved statistics cannot be reproduced.

**Interpretation:** A snapshot contributes once, regardless of the number of eligible positions, so longer coverage no longer increases its preprocessing weight. Patients with more snapshots can still contribute more; this is snapshot weighting, not equal patient weighting. Padding is excluded from fitting. Missing feature values within eligible positions are imputed rather than making the whole timestep invalid.

In [ ]:
def fit_temporal_preprocessor(X, valid, rows):
    """One vector per eligible TRAIN snapshot; repeated covered periods do not change fit weight."""
    require(X.ndim == 3 and valid.shape == X.shape[:2] and valid.dtype == bool, 'Invalid input/mask alignment.')
    require(not np.isinf(X).any() and np.isfinite(X[~valid]).all() and np.all(X[~valid] == 0),
            'Invalid input or nonzero/nonfinite padding.')
    rows = np.asarray(rows)
    require(rows.ndim == 1 and np.issubdtype(rows.dtype, np.integer) and len(rows) > 0
            and len(np.unique(rows)) == len(rows) and (rows >= 0).all() and (rows < len(X)).all(),
            'TRAIN rows must be unique valid snapshot indices.')
    eligible = rows[valid[rows].any(axis=1)]
    require(len(eligible) > 0, 'No eligible TRAIN snapshots for preprocessing.')
    first = valid[eligible].argmax(axis=1)
    observed = X[eligible, first, :]
    repeated = np.broadcast_to(observed[:, None, :], X[eligible].shape)
    require(np.array_equal(X[eligible][valid[eligible]], repeated[valid[eligible]], equal_nan=True),
            'Snapshot-only feature values must be constant across valid timesteps.')
    require(not np.isnan(observed).all(axis=0).any(),
            'A feature is missing for every eligible TRAIN snapshot; resolve source values before modeling.')
    return fit_preprocessor(observed)


def transform_temporal(X, valid, state):
    require(X.ndim == 3 and valid.shape == X.shape[:2] and valid.dtype == bool, 'Invalid input/mask alignment.')
    require(not np.isinf(X).any() and np.isfinite(X[~valid]).all(), 'Invalid input or nonfinite padding.')
    # Do not standardize placeholders; transform only genuinely available timesteps.
    values = np.zeros(X.shape, dtype=np.float32)
    if valid.any():
        values[valid], _ = transform_features(X[valid], state)
    require(np.isfinite(values).all(), 'Nonfinite Transformer input.')
    return values


def load_experiment():
    metadata, features, bundles, manifest = load_prepared()
    blobs = read_artifacts(SPLIT_TABLE, SPLIT_NAMES)
    audit = json.loads(blobs['audit.json'])
    states = json.loads(blobs['preprocessing.json'])
    frozen = read_table(PREFIX + '_PATIENT_SPLIT').select('PATIENT_ID', 'END_DT', 'RESP', 'SPLIT').toPandas()
    metadata, hashes = bind_split(metadata, frozen)
    split_statistics(metadata)
    require(hashes['snapshot_manifest_sha256'] == manifest['frozen_split_assignments_sha256'],
            'Frozen split assignments changed since preparation.')
    require(hashes == audit['source_split_hashes'] and digest_json(manifest) == audit['manifest_sha256'],
            'Saved split audit changed.')
    records = [[r.PATIENT_ID, r.END_DT, int(r.RESP), r.SPLIT] for r in metadata.itertuples()]
    require(records == json.loads(blobs['split.json']), 'Saved assignments changed.')
    require(digest_json(states) == audit['preprocessing_sha256'], 'Saved preprocessing changed.')
    require(audit.get('preprocessing_policy') == 'one_vector_per_eligible_train_snapshot_shared_v1',
            'This snapshot-only experiment requires once-per-snapshot shared preprocessing.')
    require(set(states) == {'MONTHLY', 'QUARTERLY'} and states['MONTHLY'] == states['QUARTERLY'],
            'Monthly and quarterly must share identical preprocessing state.')
    indices = {s: np.flatnonzero(metadata.SPLIT.eq(s).to_numpy()) for s in ('train', 'validation', 'test')}
    experiments = {}
    for name, b in bundles.items():
        state = states[name]
        require(state == fit_temporal_preprocessor(b['raw_X'], b['valid'], indices['train']),
                'Preprocessing does not reproduce the once-per-eligible-TRAIN-snapshot fit.')
        X = transform_temporal(b['raw_X'], b['valid'], state)
        experiments[name] = dict(b, X=X, y=metadata.RESP.to_numpy(dtype=np.float32), metadata=metadata,
            indices=indices, features=features, preprocessor=state,
            hashes={'manifest_sha256': digest_json(manifest), 'snapshot_manifest_sha256': hashes['snapshot_manifest_sha256'],
                    'preprocessing_sha256': digest_json(state), 'model_input_sha256': array_hash(X),
                    'mask_sha256': array_hash(b['valid']),
                    'preprocessing_policy': 'one_vector_per_eligible_train_snapshot_shared_v1'})
    require(np.array_equal(experiments['MONTHLY']['y'], experiments['QUARTERLY']['y']), 'Representation labels differ.')
    return experiments, manifest, audit


def fit_preprocessor(X_train):
    if X_train.ndim != 2 or not len(X_train) or np.isinf(X_train).any():
        raise ValueError("Invalid training feature matrix.")
    all_missing = np.isnan(X_train).all(axis=0)
    median = np.array([0.0 if missing else np.nanmedian(X_train[:, i])
                       for i, missing in enumerate(all_missing)])
    filled = np.where(np.isnan(X_train), median, X_train)
    mean = filled.mean(axis=0)
    scale = filled.std(axis=0)
    scale[scale == 0] = 1.0
    if not np.isfinite(np.r_[median, mean, scale]).all():
        raise ValueError("Nonfinite preprocessing statistics.")
    return {"median": median.tolist(), "mean": mean.tolist(), "scale": scale.tolist(),
            "all_missing_train": all_missing.tolist()}


def transform_features(X, state):
    if X.ndim != 2 or X.shape[1] != len(state["median"]) or np.isinf(X).any():
        raise ValueError("Feature shape or values changed.")
    mask = np.isnan(X)
    values = ((np.where(mask, state["median"], X) - state["mean"]) / state["scale"]).astype(np.float32)
    if not np.isfinite(values).all():
        raise ValueError("Nonfinite standardized values.")
    return values, mask.astype(np.float32)


### Transformer, padding masks, and inference helpers

**Why:** Define the model: 49 inputs project into the hidden dimension, receive learned position embeddings, pass through masked encoder layers, then masked mean pooling and a binary prediction head.

**Expected output:** None during definition. Shape, mask, finite-value, and patient-partition checks run when the model is used.

**Interpretation:** `~valid` prevents padded positions acting as attention keys; pooling separately excludes padded outputs. All-padded snapshots bypass the encoder and use a zero pooled vector through the learned head, so their evaluation scores carry no patient-specific feature signal. All eligible feature vectors are repeated snapshots; learned position/coverage effects cannot be described as clinical temporal evolution. Attention operates across positions, not as a direct 49-feature importance ranking.

In [ ]:
@dataclass(frozen=True)
class ModelConfig:
    input_dim: int
    seq_len: int = 12
    d_model: int = 128
    n_heads: int = 4
    encoder_layers: int = 2
    feedforward_dim: int = 256
    dropout: float = 0.2

    def __post_init__(self):
        sizes = (self.input_dim, self.seq_len, self.d_model, self.n_heads,
                 self.encoder_layers, self.feedforward_dim)
        if any(not isinstance(n, int) or isinstance(n, bool) or n <= 0 for n in sizes):
            raise ValueError("All model dimensions must be positive integers.")
        if self.d_model % self.n_heads:
            raise ValueError("d_model must be divisible by n_heads.")
        if not 0 <= self.dropout < 1:
            raise ValueError("dropout must be in [0, 1).")


class ClaimsTransformer(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.projection = nn.Linear(config.input_dim, config.d_model)
        self.position = nn.Parameter(torch.empty(1, config.seq_len, config.d_model))
        nn.init.normal_(self.position, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=config.d_model, nhead=config.n_heads,
            dim_feedforward=config.feedforward_dim, dropout=config.dropout,
            activation="relu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            layer, num_layers=config.encoder_layers, enable_nested_tensor=False,
        )
        # TransformerEncoder clones the initial layer; initialize matrix weights
        # independently so the layers do not begin with identical weights.
        for encoder_layer in self.encoder.layers:
            for parameter in encoder_layer.parameters():
                if parameter.dim() > 1:
                    nn.init.xavier_uniform_(parameter)
        self.norm = nn.LayerNorm(config.d_model)
        self.head = nn.Sequential(
            nn.Linear(config.d_model, 64), nn.ReLU(),
            nn.Dropout(config.dropout), nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
        expected = (self.config.seq_len, self.config.input_dim)
        if x.ndim != 3 or tuple(x.shape[1:]) != expected or not x.is_floating_point():
            raise ValueError(f"Expected floating [batch, {expected[0]}, {expected[1]}] input.")
        if valid.dtype != torch.bool or tuple(valid.shape) != tuple(x.shape[:2]):
            raise ValueError("Validity must be a bool [batch, timestep] tensor; True means observed.")
        if valid.device != x.device or not torch.isfinite(x).all():
            raise ValueError("Inputs/masks must share a device and all inputs must be finite.")
        # PyTorch key-padding convention: True = IGNORE. Observed zeros remain valid.
        # https://docs.pytorch.org/docs/2.8/generated/torch.nn.Transformer.html
        # All-unavailable sequences retain their rows and use the existing head on
        # zero pooled history; no artificial clinical month is made observable.
        pooled = x.new_zeros((len(x), self.config.d_model))
        active = valid.any(dim=1)
        if active.any():
            row_valid = valid[active]
            clean = x[active].masked_fill(~row_valid.unsqueeze(-1), 0.)
            hidden = self.projection(clean) + self.position
            hidden = self.norm(self.encoder(hidden, src_key_padding_mask=~row_valid))
            # Key padding does not remove padded query outputs: exclude them here.
            hidden = hidden.masked_fill(~row_valid.unsqueeze(-1), 0.)
            pooled[active] = hidden.sum(dim=1) / row_valid.sum(dim=1, keepdim=True)
        return self.head(pooled).squeeze(-1)


def seed_everything(seed: int) -> None:
    if not isinstance(seed, int) or isinstance(seed, bool) or not 0 <= seed < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32).")
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def resolve_device(device: str = "auto") -> torch.device:
    if device == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    resolved = torch.device(device)
    if resolved.type not in {"cpu", "cuda"}:
        raise ValueError("Supported devices are auto, cpu, or cuda[:index].")
    if resolved.type == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA requested but unavailable; set device='cpu'.")
    return resolved


def runtime_metadata() -> dict:
    return {
        "python_version": platform.python_version(),
        "numpy_version": str(np.__version__),
        "sklearn_version": str(sklearn.__version__),
        "torch_version": str(torch.__version__),
        "cuda_version": str(torch.version.cuda),
        "deterministic_algorithms_requested": torch.are_deterministic_algorithms_enabled(),
        "determinism_scope": "Best effort within a fixed device and software environment; unsupported operations warn.",
    }


def _primitive_state(state):
    return json.loads(json.dumps(state, sort_keys=True, allow_nan=False))


def validate_model_inputs(data, require_splits=True):
    X, valid = data["X"], data["valid"]
    representation = data["representation"]
    steps = {"MONTHLY": 12, "QUARTERLY": 4}.get(representation)
    if steps is None or X.shape != (len(data["metadata"]), steps, len(data["features"])):
        raise ValueError("Representation must be MONTHLY [N,12,F] or QUARTERLY [N,4,F] for the selected F features.")
    if X.dtype != np.float32 or valid.dtype != np.bool_ or valid.shape != X.shape[:2]:
        raise ValueError("Expected float32 tensor and aligned boolean validity mask.")
    if not np.isfinite(X).all() or not np.all(X[~valid] == 0):
        raise ValueError("Inputs must be finite; padding must remain zero after preprocessing.")
    features = data["features"]
    if not features or len(set(features)) != len(features) or "RESP" in [f.upper() for f in features]:
        raise ValueError("Unique retained predictors excluding RESP are required.")
    if not all(isinstance(f, str) and f for f in features):
        raise ValueError("Feature names must be nonempty strings.")
    meta = data["metadata"]
    if meta.duplicated(["PATIENT_ID", "END_DT"]).any():
        raise ValueError("Duplicate snapshot keys.")
    y = np.asarray(data["y"])
    if y.shape != (len(X),) or not np.isin(y, [0, 1]).all() or not np.array_equal(y, meta.RESP.to_numpy()):
        raise ValueError("Labels must match the original snapshot manifest exactly.")
    if not isinstance(data["hashes"], dict) or not data["hashes"]:
        raise ValueError("Validated source fingerprints are required.")
    _primitive_state(data["hashes"])
    _primitive_state(data["preprocessor"])
    if require_splits:
        indices = data["indices"]
        if set(indices) != {"train", "validation", "test"}:
            raise ValueError("All original patient split assignments are required.")
        seen = np.zeros(len(X), dtype=np.int8)
        sets = {}
        for split in ("train", "validation", "test"):
            rows = np.asarray(indices[split])
            if rows.ndim != 1 or not np.issubdtype(rows.dtype, np.integer) or not len(rows):
                raise ValueError("Each split requires nonempty integer snapshot indices.")
            if (rows < 0).any() or (rows >= len(X)).any() or len(np.unique(rows)) != len(rows):
                raise ValueError("Invalid or duplicate split indices.")
            seen[rows] += 1
            if not meta.iloc[rows].SPLIT.eq(split).all():
                raise ValueError("Loader assignments differ from the frozen split.")
            sets[split] = set(meta.iloc[rows].PATIENT_ID)
        if not np.all(seen == 1) or any(sets[a] & sets[b] for a, b in (("train", "validation"), ("train", "test"), ("validation", "test"))):
            raise ValueError("Population coverage or patient split separation failed.")
    return True


class SequenceDataset(Dataset):
    def __init__(self, data, split):
        self.data, self.indices = data, np.asarray(data["indices"][split])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        row = int(self.indices[index])
        x = torch.from_numpy(np.array(self.data["X"][row], dtype=np.float32, copy=True))
        valid = torch.from_numpy(np.array(self.data["valid"][row], dtype=bool, copy=True))
        y = torch.tensor(float(self.data["y"][row]), dtype=torch.float32)
        return x, valid, y


def make_loader(data, split, batch_size, seed, shuffle=False):
    return DataLoader(SequenceDataset(data, split), batch_size=batch_size,
                      shuffle=shuffle, num_workers=0, drop_last=False,
                      generator=torch.Generator().manual_seed(seed))


def predict_loader(model, loader, device, criterion=None):
    model.eval()
    labels, scores, total_loss, count = [], [], 0.0, 0
    with torch.inference_mode():
        for x, valid, y in loader:
            x, valid, y = x.to(device), valid.to(device), y.to(device)
            logits = model(x, valid)
            if not torch.isfinite(logits).all():
                raise ValueError("Nonfinite model predictions.")
            if criterion is not None:
                loss = criterion(logits, y)
                if not torch.isfinite(loss):
                    raise ValueError("Nonfinite evaluation loss.")
                total_loss += float(loss.item()) * len(y)
            count += len(y)
            labels.append(y.cpu().numpy())
            scores.append(torch.sigmoid(logits).cpu().numpy())
    if not count:
        raise ValueError("Cannot score an empty split.")
    return total_loss / count, np.concatenate(labels), np.concatenate(scores)


### Ranking metrics and validation-selected threshold

**Why:** Define average precision (AP), ROC AUC, threshold-dependent errors, decile lift, and top-percent capture. Choose the classification threshold using validation F1 only; keep it frozen for later evaluation.

**Expected output:** No tables until the functions are called. Subsequent reports show how well positive snapshots concentrate near the top of the model's ranking.

**Interpretation:** Higher AP and top-10% lift are better; lift 1 is the partition's baseline and lift 3 means three times its average positive rate in the selected group. AP is sklearn average precision, not trapezoidal PR area. Weighted training scores are not automatically calibrated probabilities. Metrics and confusion counts are snapshot-weighted; repeated patients limit independence.

In [ ]:
def _validate(y, probabilities):
    labels = np.asarray(y)
    raw_scores = np.asarray(probabilities)
    if np.iscomplexobj(raw_scores) or (
        raw_scores.dtype == object
        and any(isinstance(value, (complex, np.complexfloating)) for value in raw_scores.flat)
    ):
        raise ValueError("Probabilities must be real values, not complex numbers.")
    scores = np.asarray(raw_scores, dtype=float)
    if labels.ndim != 1 or scores.ndim != 1 or len(labels) != len(scores) or not len(labels):
        raise ValueError("Labels and probabilities must be aligned, nonempty 1-D arrays.")
    if not np.isin(labels, [0, 1]).all():
        raise ValueError("Labels must be binary 0/1.")
    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError("Probabilities must be finite and in [0, 1].")
    return labels.astype(np.int64), scores


def select_validation_threshold(y, probabilities) -> float:
    """Maximize VALIDATION F1; an exact tie uses the highest threshold.

    The caller must provide VALIDATION labels and scores, never TEST. Predictions
    are positive when score >= threshold. Equal scores are never split, and
    integer cross-products identify exact F1 ties without rounding ambiguity.
    """
    labels, scores = _validate(y, probabilities)
    if len(np.unique(labels)) != 2:
        raise ValueError("Threshold selection requires both VALIDATION classes.")
    order = np.argsort(scores, kind="stable")[::-1]
    ranked_scores = scores[order]
    true_positives = np.cumsum(labels[order], dtype=np.int64)
    group_ends = np.r_[np.flatnonzero(ranked_scores[:-1] != ranked_scores[1:]), len(labels) - 1]
    total_positives = int(labels.sum())
    best_numerator, best_denominator = 0, 1
    best_threshold = float(ranked_scores[0])
    for end in group_ends:
        # F1 = 2 TP / (number selected + total positives). Python integers
        # keep cross-products exact and avoid fixed-width integer overflow.
        numerator = 2 * int(true_positives[end])
        denominator = int(end) + 1 + total_positives
        if numerator * best_denominator > best_numerator * denominator:
            best_numerator, best_denominator = numerator, denominator
            best_threshold = float(ranked_scores[end])
        # Descending thresholds retain the highest cutoff on an exact tie.
    return best_threshold


def classification_metrics(y, probabilities, threshold: float) -> dict:
    labels, scores = _validate(y, probabilities)
    if not np.isfinite(threshold) or not 0 <= threshold <= 1:
        raise ValueError("The fixed classification threshold must be in [0, 1].")
    predicted = (scores >= threshold).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    return {
        "average_precision": float(average_precision_score(labels, scores)) if labels.sum() else None,
        "roc_auc": float(roc_auc_score(labels, scores)) if len(np.unique(labels)) == 2 else None,
        "precision": float(precision_score(labels, predicted, zero_division=0)),
        "recall": float(recall_score(labels, predicted, zero_division=0)),
        "f1": float(f1_score(labels, predicted, zero_division=0)),
        "threshold": float(threshold),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def rank_tables(metadata, scores):
    labels, scores = _validate(metadata.RESP.to_numpy(), scores)
    if metadata.duplicated(["PATIENT_ID", "END_DT"]).any() or metadata[["PATIENT_ID", "END_DT"]].isna().any().any():
        raise ValueError("Ranking requires unique nonnull snapshot keys.")
    ranked = metadata[["PATIENT_ID", "END_DT", "RESP"]].reset_index(drop=True).copy()
    ranked["SCORE"] = scores
    ranked = ranked.sort_values(["SCORE", "PATIENT_ID", "END_DT"], ascending=[False, True, True]).reset_index(drop=True)
    n, positives = len(ranked), int(labels.sum())
    base = positives / n
    ranked["DECILE"] = 10 - np.minimum(9, np.arange(n) * 10 // n)
    deciles = []
    cumulative_n = cumulative_positive = 0
    for decile in range(10, 0, -1):
        part = ranked[ranked.DECILE == decile]
        if part.empty:
            continue
        count, positive = len(part), int(part.RESP.sum())
        cumulative_n += count
        cumulative_positive += positive
        rate = positive / count
        deciles.append({"decile": decile, "snapshots": count, "positives": positive,
                        "score_min": float(part.SCORE.min()), "score_max": float(part.SCORE.max()),
                        "response_rate": rate, "lift": rate / base if base else None,
                        "cumulative_snapshots": cumulative_n, "cumulative_positives": cumulative_positive,
                        "cumulative_lift": (cumulative_positive / cumulative_n) / base if base else None,
                        "cumulative_recall": cumulative_positive / positives if positives else None})
    top = []
    for fraction in (.05, .10, .20, .30):
        k = max(1, int(np.ceil(n * fraction)))
        tp = int(ranked.RESP.iloc[:k].sum())
        top.append({"fraction": fraction, "observations_evaluated": n,
                    "baseline_positive_rate": base, "selected": k,
                    "positives": tp, "total_positives": positives,
                    "positive_rate_in_selected": tp / k, "resp1_captured": tp,
                    "precision": tp / k, "recall": tp / positives if positives else None,
                    "lift": (tp / k) / base if base else None})
    return pd.DataFrame(deciles), pd.DataFrame(top)


def top10_lift(y, scores, metadata):
    labels, scores = _validate(y, scores)
    if not np.array_equal(labels, metadata.RESP.to_numpy()) or not 0 < labels.sum() < len(labels):
        raise ValueError("Lift requires aligned metadata and both classes.")
    _, top = rank_tables(metadata, scores)
    return float(top.loc[top.fraction.eq(.10), "lift"].iloc[0])


def evaluate_partition(model, payload, data, split, device):
    rows = data['indices'][split]
    _, y, scores = predict_loader(model, make_loader(data, split, TRAINING_SETTINGS['batch_size'], 42), device)
    meta = data['metadata'].iloc[rows].reset_index(drop=True)
    require(np.array_equal(y, meta.RESP.to_numpy()), 'Prediction labels do not align with snapshot keys.')
    deciles, top = rank_tables(meta, scores)
    metrics = classification_metrics(y, scores, payload['validation_threshold'])
    top10 = top.loc[top.fraction.eq(.1)].iloc[0]
    metrics.update(model=data['representation'], split=split, snapshots=len(y), positives=int(y.sum()),
        top10_lift=float(top10['lift']), top10_precision=float(top10['precision']), top10_recall=float(top10['recall']))
    if split == 'validation':
        require(np.isclose(metrics['average_precision'], payload['best_validation_average_precision'], rtol=0, atol=1e-7), 'Saved validation AP failed reproduction.')
    return {'metrics': metrics, 'deciles': deciles, 'top': top}


### Explicit L1/L2, early stopping, and checkpoint verification

**Why:** Define the optimization objective as weighted binary cross-entropy plus `L1=lambda1*sum(abs(W))` and `L2=0.5*lambda2*sum(W**2)`. Penalties cover trainable linear/attention weight matrices; biases, LayerNorm vectors, and position embeddings are excluded. AdamW weight decay stays zero when explicit L2 is enabled.

**Expected output:** No training starts in this definition cell. Training later logs separate penalty terms, AP, losses, and lift, and restores the epoch with highest validation AP.

**Interpretation:** TRAIN-derived positive-class weighting addresses label imbalance, not feature priority. L1/L2 strengths are configured, not automatically tuned here. Early stopping limits deterioration but cannot guarantee no overfitting. Loss plots exclude penalty terms and are computed in evaluation mode, making TRAIN/VALIDATION comparable. TEST never drives updates, stopping, or threshold selection.

In [ ]:
def validate_regularization_settings(settings):
    for name in ("l1_lambda", "l2_lambda", "weight_decay"):
        value = settings[name]
        if isinstance(value, (bool, np.bool_)) or not np.isfinite(value) or value < 0:
            raise ValueError(f"{name} must be nonnegative and finite.")
    if settings["regularization_scope"] != "linear_and_attention_weights":
        raise ValueError("Unsupported regularization scope.")
    if settings["l2_lambda"] > 0 and settings["weight_decay"] > 0:
        raise ValueError("Use explicit L2 or AdamW weight decay, not both.")


def regularization_penalties(model, l1_lambda, l2_lambda):
    """Explicit penalties; exclude biases, LayerNorm vectors and position embeddings."""
    weights = [p for name, p in model.named_parameters()
               if p.requires_grad and p.ndim > 1 and name.endswith("weight")]
    if not weights:
        raise ValueError("No eligible trainable weight matrices for regularization.")
    zero = weights[0].new_zeros(())
    l1 = l1_lambda * sum((p.abs().sum() for p in weights), zero) if l1_lambda else zero
    l2 = .5 * l2_lambda * sum((p.square().sum() for p in weights), zero) if l2_lambda else zero
    return l1, l2


def train_run(data, model_config, settings, run_id):
    validate_regularization_settings(settings)
    validate_model_inputs(data)
    if tuple(data["X"].shape[1:]) != (model_config.seq_len, model_config.input_dim):
        raise ValueError("Architecture and tensor dimensions disagree.")
    for name in ("epochs", "patience", "batch_size"):
        if type(settings[name]) is not int or settings[name] <= 0:
            raise ValueError(f"{name} must be a positive integer.")
    for name in ("learning_rate", "grad_clip"):
        if not np.isfinite(settings[name]) or settings[name] <= 0:
            raise ValueError(f"{name} must be positive and finite.")
    for name in ("weight_decay", "min_delta"):
        if not np.isfinite(settings[name]) or settings[name] < 0:
            raise ValueError(f"{name} must be nonnegative and finite.")
    seed_everything(settings["seed"])
    device = resolve_device(settings["device"])
    class_counts = {}
    for name in ("train", "validation"):
        y = data["y"][data["indices"][name]]
        positives = int(y.sum())
        negatives = len(y) - positives
        if not positives or not negatives:
            raise ValueError(f"{name} requires both classes.")
        class_counts[name] = {"snapshots": len(y), "positives": positives, "negatives": negatives}
    pos_weight = class_counts["train"]["negatives"] / class_counts["train"]["positives"]
    model = ClaimsTransformer(model_config).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=settings["learning_rate"], weight_decay=settings["weight_decay"])
    train_loader = make_loader(data, "train", settings["batch_size"], settings["seed"], True)
    validation_loader = make_loader(data, "validation", settings["batch_size"], settings["seed"])
    diagnostic_loader = make_loader(data, "train", settings["batch_size"], settings["seed"])
    best_ap, patience_reference = -np.inf, -np.inf
    without_progress, best_epoch, best_state = 0, None, None
    history = []
    stop_reason, early_stopping_epoch = "maximum_epochs", None
    print(f'{data["representation"]}: training on {device}; TRAIN positive weight={pos_weight:.4f}.', flush=True)
    for epoch in range(1, settings["epochs"] + 1):
        model.train()
        loss_sum, bce_sum, l1_sum, l2_sum, count = 0.0, 0.0, 0.0, 0.0, 0
        for x, valid, y in train_loader:
            x, valid, y = x.to(device), valid.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            data_loss = criterion(model(x, valid), y)
            l1_penalty, l2_penalty = regularization_penalties(model, settings["l1_lambda"], settings["l2_lambda"])
            loss = data_loss + l1_penalty + l2_penalty
            if not torch.isfinite(loss):
                raise ValueError("Nonfinite training loss.")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), settings["grad_clip"], error_if_nonfinite=True)
            optimizer.step()
            loss_sum += float(loss.item()) * len(y)
            bce_sum += float(data_loss.item()) * len(y)
            l1_sum += float(l1_penalty.item()) * len(y)
            l2_sum += float(l2_penalty.item()) * len(y)
            count += len(y)
        val_loss, val_y, val_scores = predict_loader(model, validation_loader, device, criterion)
        val_ap = float(average_precision_score(val_y, val_scores))
        train_loss, train_y, train_scores = predict_loader(model, diagnostic_loader, device, criterion)
        train_ap = float(average_precision_score(train_y, train_scores))
        train_lift = top10_lift(train_y, train_scores, data["metadata"].iloc[data["indices"]["train"]])
        val_lift = top10_lift(val_y, val_scores, data["metadata"].iloc[data["indices"]["validation"]])
        history.append({"epoch": epoch, "optimization_loss": loss_sum / count,
                        "optimization_bce": bce_sum / count, "optimization_l1": l1_sum / count,
                        "optimization_l2": l2_sum / count,
                        "training_loss": train_loss, "validation_loss": val_loss,
                        "training_top10_lift": train_lift, "validation_top10_lift": val_lift,
                        "training_average_precision": train_ap, "validation_average_precision": val_ap,
                        "validation_roc_auc": float(roc_auc_score(val_y, val_scores))})
        if val_ap > best_ap:
            best_ap, best_epoch = val_ap, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if val_ap > patience_reference + settings["min_delta"]:
            patience_reference, without_progress = val_ap, 0
        else:
            without_progress += 1
        print(f'Epoch {epoch:02d}: train loss={train_loss:.5f}; validation loss={val_loss:.5f}; '
              f'TRAIN AP={train_ap:.5f}; VALIDATION AP={val_ap:.5f}; '
              f'TRAIN lift={train_lift:.3f}; VALIDATION lift={val_lift:.3f}', flush=True)
        if without_progress >= settings["patience"]:
            stop_reason, early_stopping_epoch = "validation_AP_patience", epoch
            print(f"Early stopping; restoring epoch {best_epoch}.", flush=True)
            break
    if best_state is None:
        raise ValueError("Training did not produce a valid checkpoint.")
    model.load_state_dict(best_state, strict=True)
    _, val_y, val_scores = predict_loader(model, validation_loader, device, criterion)
    if not np.isclose(average_precision_score(val_y, val_scores), best_ap, rtol=0, atol=1e-7):
        raise ValueError("Restored checkpoint failed validation AP reproduction.")
    threshold = select_validation_threshold(val_y, val_scores)
    state = _primitive_state(data["preprocessor"])
    summary = {"run_id": run_id, "representation": data["representation"], "training_complete": True,
               "completed_at_utc": datetime.now(timezone.utc).isoformat(),
               "best_epoch": int(best_epoch), "epochs_completed": len(history),
               "early_stopping_epoch": early_stopping_epoch, "stop_reason": stop_reason,
               "best_validation_average_precision": best_ap, "validation_threshold": threshold,
               "validation_metrics": classification_metrics(val_y, val_scores, threshold),
               "train_pos_weight": float(pos_weight), "class_counts": class_counts,
               "model_parameter_count": sum(p.numel() for p in model.parameters()),
               "model_config": asdict(model_config), "training_settings": dict(settings),
               "input_hashes": _primitive_state(data["hashes"]), "resolved_device": str(device),
               "preprocessor": state, "time_orientation": TIME_ORIENTATION, "mask_convention": MASK_CONVENTION,
               "all_padded_snapshots": int((~data["valid"].any(axis=1)).sum()),
               "preprocessing": "Already transformed with shared once-per-eligible-TRAIN-snapshot preprocessing; no additional encoding",
               "test_inference_performed": False, "runtime": runtime_metadata()}
    payload = {"format_version": 2, "training_complete": True, "run_id": run_id,
               "representation": data["representation"], "model_state_dict": best_state,
               "model_config": asdict(model_config), "training_settings": _primitive_state(settings),
               "input_hashes": _primitive_state(data["hashes"]), "feature_names": list(data["features"]),
               "time_steps": list(range(model_config.seq_len)), "time_orientation": TIME_ORIENTATION,
               "mask_convention": MASK_CONVENTION, "preprocessor": state,
               "tensor_shape": [int(v) for v in data["X"].shape],
               "transform": "frozen_train_preprocessor", "validation_threshold": float(threshold),
               "selection_metric": "validation_average_precision", "best_epoch": int(best_epoch),
               "best_validation_average_precision": best_ap}
    buffer = io.BytesIO()
    torch.save(payload, buffer)
    return buffer.getvalue(), summary, pd.DataFrame(history)


def load_verified_model(blob, data, run_id, device="auto", expected_settings=None, expected_config=None):
    validate_model_inputs(data)
    payload = torch.load(io.BytesIO(blob), map_location="cpu", weights_only=True)
    if (not isinstance(payload, dict) or payload.get("format_version") != 2
            or payload.get("training_complete") is not True or payload.get("run_id") != run_id):
        raise ValueError("Checkpoint is incomplete, unsupported, or belongs to another run.")
    expected = {"input_hashes": _primitive_state(data["hashes"]),
                "feature_names": list(data["features"]), "representation": data["representation"],
                "preprocessor": _primitive_state(data["preprocessor"]),
                "time_steps": list(range(data["X"].shape[1])), "tensor_shape": list(data["X"].shape),
                "time_orientation": TIME_ORIENTATION, "mask_convention": MASK_CONVENTION,
                "transform": "frozen_train_preprocessor", "selection_metric": "validation_average_precision"}
    for key, value in expected.items():
        if payload.get(key) != value:
            raise ValueError(f"Checkpoint differs from verified inputs on {key}.")
    if expected_settings is not None and payload.get("training_settings") != _primitive_state(expected_settings):
        raise ValueError("Checkpoint training settings differ from declared settings.")
    if expected_config is not None:
        expected_config = asdict(expected_config) if isinstance(expected_config, ModelConfig) else dict(expected_config)
        if payload.get("model_config") != expected_config:
            raise ValueError("Checkpoint architecture differs from declared settings.")
    threshold = payload.get("validation_threshold")
    if threshold is None or not np.isfinite(threshold) or not 0 <= threshold <= 1:
        raise ValueError("Missing valid frozen validation threshold.")
    model = ClaimsTransformer(ModelConfig(**payload["model_config"]))
    if (model.config.seq_len, model.config.input_dim) != tuple(data["X"].shape[1:]):
        raise ValueError("Checkpoint architecture is inconsistent with tensor layout.")
    model.load_state_dict(payload["model_state_dict"], strict=True)
    resolved = resolve_device(device)
    model.to(resolved).eval()
    return model, payload, resolved


def plot_history(history, title):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    history.plot(x="epoch", y=["training_loss", "validation_loss"], ax=axes[0], title=title + " weighted BCE (eval mode)")
    history.plot(x="epoch", y=["training_average_precision", "validation_average_precision"], ax=axes[1], title=title + " AP")
    history.plot(x="epoch", y=["training_top10_lift", "validation_top10_lift"], ax=axes[2], title=title + " top-10% lift")
    fig.tight_layout()
    return fig


def model_contract(data, config):
    return {'consumer_implementation_sha256': CONSUMER_IMPLEMENTATION_SHA256, 'input_hashes': data['hashes'],
            'representation': data['representation'], 'config': asdict(config), 'training': TRAINING_SETTINGS}


def restored_representation(name, data):
    config = ModelConfig(input_dim=len(data['features']), seq_len=data['X'].shape[1], **MODEL_SETTINGS)
    blobs = read_artifacts(RUN_PREFIX + '_' + name + '_MODEL', MODEL_NAMES)
    summary = json.loads(blobs['summary.json'])
    require(summary['contract'] == model_contract(data, config), 'Saved checkpoint contract differs.')
    model, payload, device = load_verified_model(blobs['checkpoint.pt'], data, RUN_ID + '_' + name,
        expected_settings=TRAINING_SETTINGS, expected_config=config)
    require(summary['best_epoch'] == payload['best_epoch'] and summary['best_validation_average_precision'] == payload['best_validation_average_precision'], 'Summary/checkpoint selection differs.')
    return model, payload, device, summary, pd.DataFrame(json.loads(blobs['history.json']))


### Verify inputs immediately before training

**Why:** Reload the approved artifacts, validate regularization settings, and confirm the network receives the expected snapshot-only tensors and masks.

**Expected output:** Declared L1/L2 values and scope, MONTHLY `[23151,12,49]`, QUARTERLY `[23151,4,49]`, corresponding mask shapes, all-padded counts, and the 49-feature source audit.

**Interpretation:** Any contract error stops the run before optimization. Snapshot broadcasting supplies no within-snapshot feature trends; the two models share preprocessing and differ in mask granularity, sequence length, and learned parameters. A nonzero all-padded count identifies a subgroup without feature-informed predictions. Printed upstream encoding limitations still apply even when local scaling and splitting checks pass.

In [ ]:
# Load fixed inputs and verify controlled comparison
experiments = manifest = split_audit = None
experiments, manifest, split_audit = load_experiment()
validate_regularization_settings(TRAINING_SETTINGS)
print('Explicit regularization:', {k: TRAINING_SETTINGS[k] for k in ('l1_lambda', 'l2_lambda', 'weight_decay', 'regularization_scope')})
for name, data in experiments.items():
    validate_model_inputs(data)
    print(name, data['X'].shape, '| valid mask:', data['valid'].shape, '| all-padded snapshots:', int((~data['valid'].any(axis=1)).sum()))
print('PyTorch boolean src_key_padding_mask: True = ignore; passed as ~valid. Pooling also excludes padded positions.')
print('All-padded snapshots remain in the population; encoder bypass, zero pooled history through unchanged head.')
print('Input basis: 49 repeated V63 snapshot features -> enrollment-overlap availability -> shared snapshot-level TRAIN imputation/scaling -> attention/pooling mask.')
print('Retrospective limitation:', manifest['business_encoding']['upstream_fit_population'])



print('Input design:', manifest['input_description'])
display(pd.DataFrame(manifest['audit'])[['FEATURE_NAME', 'VALUE_SOURCE']])


### Train the two representations and inspect learning curves

**Why:** Train fresh MONTHLY and QUARTERLY models under the same declared settings, save each best-validation checkpoint, reload it, and plot the training history. A matching artifact from this same experiment can be verified on rerun; different contracts are rejected.

**Expected output:** Epoch logs, best/restored epoch, stop reason, and three plots: evaluation-mode weighted BCE, AP, and top-10% lift.

**Interpretation:** Rising TRAIN AP while VALIDATION AP stalls/falls suggests overfitting; rising validation loss strengthens that concern. If both metrics remain weak, consider insufficient signal or unsuitable model complexity. A smaller train-validation gap alone does not establish improvement—validation performance must also improve. These curves cannot identify the responsible features or prove a cause; notebook 04 adds feature, coverage, and controlled-model comparisons.

In [ ]:
# Train MONTHLY then QUARTERLY; save separate best-validation checkpoints
for name in ('MONTHLY', 'QUARTERLY'):
    data = experiments[name]
    config = ModelConfig(input_dim=len(data['features']), seq_len=data['X'].shape[1], **MODEL_SETTINGS)
    table = RUN_PREFIX + '_' + name + '_MODEL'
    if not table_exists(table):
        checkpoint, summary, history = train_run(data, config, TRAINING_SETTINGS, RUN_ID + '_' + name)
        summary['contract'] = model_contract(data, config)
        save_artifacts(table, {'checkpoint.pt': checkpoint, 'summary.json': canonical_json(summary).encode(),
                              'history.json': canonical_json(history.to_dict('records')).encode()})
    model, payload, device, summary, history = restored_representation(name, data)
    print(name, '| best epoch:', summary['best_epoch'], '| stopped at:', summary['epochs_completed'],
          '| early stopping epoch:', summary['early_stopping_epoch'], '| reason:', summary['stop_reason'])
    import matplotlib.pyplot as plt
    plot_history(history, name)
    plt.show()
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


### Compare restored checkpoints without using TEST

**Why:** Evaluate each selected checkpoint on TRAIN and VALIDATION and reproduce the saved validation AP, ensuring displayed metrics correspond to the restored model rather than its final training epoch.

**Expected output:** Four rows: two representations times two partitions, including AP, classification metrics, top-10% precision/recall/lift, and counts. No TEST predictions are made by this cell.

**Interpretation:** Prefer a model with stronger validation evidence and manageable generalization gap, not simply the highest TRAIN score. Small differences can reflect sampling variability; do not claim superiority without uncertainty analysis. Improvements here cannot demonstrate a benefit from historical sequence information because all 49 inputs are repeated snapshot values. Feature ranking, correlations, and overfitting diagnostics are in notebook 04.

In [ ]:
# Display TRAIN and VALIDATION diagnostics at the restored checkpoints
diagnostics = []
for name in ('MONTHLY', 'QUARTERLY'):
    data = experiments[name]
    model, payload, device, summary, history = restored_representation(name, data)
    for split in ('train', 'validation'):
        report = evaluate_partition(model, payload, data, split, device)
        diagnostics.append(report['metrics'])
    del model
display(pd.DataFrame(diagnostics))
print('No TEST predictions were made. Choose settings using validation evidence only; run notebook 04 for retrospective evaluation.')
